# Over-Fitting and Model Tuning

## Initializations

In [8]:
# Required packages
import pandas as pd       # For data handling
import numpy as np        # For numerical operations (optional but useful)
import matplotlib.pyplot as plt  # For plotting
import seaborn as sns     # For nicer statistical plots
import scienceplots
from scipy.stats import skew
from scipy.cluster.hierarchy import linkage, leaves_list
from sklearn.preprocessing import PowerTransformer # For transformations 
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA # For Principal Component Analysis
from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

import sys
import os

# Absolute path to the repo root
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../"))

if repo_root not in sys.path:
    sys.path.append(repo_root)

from utils import check_transform_suitability, selective_transform, near_zero_var, find_correlation, plot_corr

# Set the plotting style
sns.set_theme(style="darkgrid")  # Set theme for seaborn plots

# Load data required for the analysis in this notebook
two_cd = pd.read_parquet("../../data/twoClassData.parquet")
german_credit = pd.read_parquet("../../data/GermanCredit.parquet")
chem_man_pro = pd.read_parquet("../../data/ChemicalManufacturingProcess.parquet")

## Investigate data

In [2]:
two_cd.head()

,classes,PredictorA,PredictorB
0,Class2,0.1582,0.1609
1,Class2,0.6552,0.4918
2,Class2,0.7060,0.6333
3,Class2,0.1992,0.0881
4,Class2,0.3952,0.4152


In [3]:
print(two_cd["classes"].unique(), "\n")      # List of unique class labels
print(two_cd["classes"].nunique())     # Number of unique classes
print(two_cd["classes"].value_counts()) # Frequency of each class

['Class2', 'Class1']
Categories (2, object): ['Class1', 'Class2'] 

2
classes
Class1    111
Class2     97
Name: count, dtype: int64


In [4]:
two_cd.describe()   # Summary statistics for numeric columns

,PredictorA,PredictorB
count,208.000000,208.000000
mean,0.250221,0.236013
std,0.140072,0.132705
min,0.023600,0.028900
25%,0.133475,0.129250
50%,0.249050,0.224800
75%,0.331250,0.301650
max,0.706000,0.734200


In [5]:
two_cd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   classes     208 non-null    category
 1   PredictorA  208 non-null    float64 
 2   PredictorB  208 non-null    float64 
dtypes: category(1), float64(2)
memory usage: 3.7 KB


## Data Splitting

In [7]:
# Classes column contains the categorical labels
X = two_cd.drop(columns=["classes"])
y = two_cd["classes"]

# Stratified split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    train_size=0.8, 
    random_state=1,   # same as set.seed(1)
    stratify=y        # ensures class proportions are preserved
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("First few training indices:", X_train.index[:6])
print("First few test indices:", X_test.index[:6])

Train shape: (166, 2) (166,)
Test shape: (42, 2) (42,)
First few training indices: Index([183, 27, 149, 4, 94, 53], dtype='int64')
First few test indices: Index([45, 23, 74, 65, 204, 108], dtype='int64')


## Basic Model Building

### Fit a 5-nearest neighbor model, knn

In [ ]:
knn_fit = KNeighborsClassifier(n_neighbors=5)
knn_fit.fit(X_train, y_train)

KNeighborsClassifier()

In [13]:
# Predict on test set
test_predictions = knn_fit.predict(X_test)
print("Test Predictions:", test_predictions)

Test Predictions: ['Class2' 'Class2' 'Class2' 'Class2' 'Class1' 'Class2' 'Class1' 'Class1'
 'Class1' 'Class1' 'Class1' 'Class2' 'Class1' 'Class1' 'Class2' 'Class1'
 'Class1' 'Class1' 'Class1' 'Class1' 'Class1' 'Class2' 'Class1' 'Class1'
 'Class2' 'Class2' 'Class2' 'Class1' 'Class1' 'Class2' 'Class1' 'Class2'
 'Class1' 'Class1' 'Class2' 'Class1' 'Class2' 'Class1' 'Class1' 'Class2'
 'Class1' 'Class1']


In [14]:
print(type(test_predictions))
print(test_predictions.shape)

<class 'numpy.ndarray'>
(42,)


### Predict class probabilities on test set

In [ ]:
# This corresponds to the “proportion of neighbors for each class” mentioned for knn3:
# “The knn3 function can produce class predictions as well as the proportion of neighbors for each class.”
test_probabilities = knn_fit.predict_proba(X_test)
print(pd.DataFrame(test_probabilities, columns=knn_fit.classes_).head())

   Class1  Class2
0     0.4     0.6
1     0.2     0.8
2     0.2     0.8
3     0.0     1.0
4     1.0     0.0
